# Multimodal Retrieval + Zero-shot 분류 — SigLIP 2

**목표**: SigLIP 2를 단일 임베딩 모델로 활용해 (1) text-to-image retrieval, (2) image-to-image 유사도, (3) zero-shot 분류 세 가지 모두 동작시키기.

**SigLIP 2가 SigLIP / CLIP 대비 갖는 차이**
- Sigmoid loss: positive와 negative를 *짝별로* 처리해 큰 batch에서 안정적
- Multilingual: 한국어 쿼리도 의미 단위로 동작 (CLIP 원판은 영어 위주)
- 기본 해상도 224/256, dense feature 품질 향상

## 0. 환경 설정

> **40분 핵심 경로**: SigLIP 2로 텍스트→이미지 검색과 영문·한국어 zero-shot 정확도를 비교합니다. PCA와 이미지→이미지 검색은 선택입니다.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModel, AutoProcessor
from datasets import load_dataset

CKPT = 'google/siglip2-base-patch16-256'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModel.from_pretrained(CKPT).to(device).eval()
processor = AutoProcessor.from_pretrained(CKPT)
print(f'Loaded {CKPT}')

## 1. 이미지 갤러리 만들기

`frgfm/imagenette` (ImageNet 10-class subset)에서 100장 추출. 클래스: tench(어류), English springer(개), cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute.

이 클래스들로 한국어 쿼리("낚시", "강아지", "교회", "낙하산" 등)를 던져 SigLIP 2의 multilingual 능력을 검증.

In [ ]:
ds = load_dataset('frgfm/imagenette', '160px', split='validation')
ds = ds.shuffle(seed=42).select(range(100))

label_names = ds.features['label'].names
print('클래스:', label_names)
print(f'갤러리 이미지 수: {len(ds)}')

# 미리보기
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, ex in zip(axes.flat, ds.select(range(10))):
    ax.imshow(ex['image']); ax.axis('off')
    ax.set_title(label_names[ex['label']], fontsize=8)
plt.tight_layout(); plt.show()

## 2. 이미지 임베딩 일괄 계산

`get_image_features`로 각 이미지의 512-d 임베딩 추출. cosine 비교를 위해 L2 정규화.

In [ ]:
@torch.no_grad()
def encode_images(images, batch_size=16):
    feats = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        inputs = processor(images=batch, return_tensors='pt').to(device)
        f = model.get_image_features(**inputs)
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu())
    return torch.cat(feats, dim=0)

@torch.no_grad()
def encode_texts(texts):
    inputs = processor(text=texts, return_tensors='pt',
                       padding='max_length', truncation=True).to(device)
    f = model.get_text_features(**inputs)
    f = f / f.norm(dim=-1, keepdim=True)
    return f.cpu()

images = [ex['image'].convert('RGB') for ex in ds]
labels = np.array([ex['label'] for ex in ds])

image_feats = encode_images(images)  # (100, 768)
print('image_feats:', image_feats.shape)

## 3. Text → Image Retrieval

텍스트 쿼리 임베딩과 갤러리 이미지 임베딩을 cosine similarity로 비교하고 top-K를 가져옴.

In [ ]:
def show_topk(query_text: str, k=5):
    qf = encode_texts([query_text])
    sims = (qf @ image_feats.T).squeeze(0)  # (100,)
    top = sims.argsort(descending=True)[:k]
    fig, axes = plt.subplots(1, k, figsize=(2.5*k, 3))
    for ax, idx in zip(axes, top):
        idx = int(idx)
        ax.imshow(images[idx]); ax.axis('off')
        ax.set_title(f'{label_names[labels[idx]]}\n{sims[idx]:.3f}', fontsize=8)
    fig.suptitle(f'Query: {query_text}', y=1.02)
    plt.tight_layout(); plt.show()

show_topk('a chain saw cutting wood', k=5)

### TODO #1 — 한국어 쿼리로 검색

다음 5개 한국어 쿼리에 대해 top-5를 시각화하고, 결과에 관해 코멘트로 기록하세요.

1. "낚시하는 사람"
2. "강아지"
3. "산 속 교회"
4. "하늘에서 떨어지는 낙하산"
5. "오래된 자동차" *(주의: imagenette에는 자동차 클래스가 없음. 어떻게 동작?)*

관찰 포인트: 5번처럼 *클래스에 없는* 쿼리는 어떻게 처리되는가? 가장 가까운 다른 클래스로 가는가?

In [ ]:
queries_ko = [
    '낚시하는 사람',
    '강아지',
    '산 속 교회',
    '하늘에서 떨어지는 낙하산',
    '오래된 자동차',
]

# TODO: 각 쿼리에 대해 show_topk(q) 를 호출해 top-5 시각화
# TODO: 결과를 보고 관찰 코멘트를 주석으로 남기기 (특히 5번 '오래된 자동차'는 클래스에 없음)

In [ ]:
# ✅ 참고 답안 (핵심)
for q in queries_ko:
    show_topk(q)
# 관찰: 1~4 는 대응 클래스(tench/springer/church/parachute)가 상위.
#       5 '오래된 자동차'는 클래스에 없어 시각적으로 가까운 차량/기계류로 매칭됨.

## 선택 실습: 임베딩 공간 시각화

768-d 임베딩을 2D로 떨어뜨려 클래스가 자연스럽게 군집을 이루는지 확인.

### TODO #2 — 임베딩 PCA 2D 시각화

이미지 임베딩(`image_feats`)을 `torch.pca_lowrank` 로 2D 로 투영하고, 클래스별로 색을 달리해 산점도(scatter)로 그려 군집이 형성되는지 확인하세요.

In [ ]:
# TODO: image_feats 를 2D 로 투영 (힌트: torch.pca_lowrank(image_feats, q=2))
emb2d = None

# TODO: label 별로 색을 달리해 scatter plot, legend 에 label_names 표시

In [ ]:
# ✅ 참고 답안 (핵심)
U, S, V = torch.pca_lowrank(image_feats, q=2)
emb2d = (U * S).numpy()                       # (100, 2)
plt.figure(figsize=(8, 6))
for cls in range(len(label_names)):
    mask = labels == cls
    if mask.sum():
        plt.scatter(emb2d[mask, 0], emb2d[mask, 1], label=label_names[cls], alpha=0.7)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.title('SigLIP 2 image embeddings (PCA 2D)'); plt.tight_layout(); plt.show()

**관찰 포인트**
- 대부분의 클래스가 뚜렷한 군집을 형성함.
- 일부 시각적으로 유사한 클래스 (예: garbage truck ↔ gas pump, 둘 다 기계/차량 관련)는 가까운 위치에 나타남.
- SigLIP 2 임베딩이 의미적 구분을 잘 학습했음을 시각적으로 확인 가능.

## 5. Zero-shot 분류

임베딩만 있으면 분류기는 사실상 따라오는 셈.
각 클래스 이름으로 텍스트 임베딩을 만들고, 이미지 임베딩과 가장 비슷한 클래스를 답으로 선택.

**SigLIP의 사용 패턴**: 일반적인 CLIP-style softmax 대신 sigmoid 점수가 자연스러움 (per-class independent).

### TODO #3 — Zero-shot 정확도 측정

1. 각 클래스 이름에 "a photo of a {class}" 프롬프트 템플릿을 적용해 10개 텍스트 임베딩 생성
2. 100개 이미지 각각에 대해 가장 유사도 높은 클래스를 예측
3. ground-truth와 비교해 accuracy 측정
4. **추가**: 한국어 프롬프트("{class}의 사진") 버전도 만들어 정확도 비교

관찰 포인트: 한국어 프롬프트와 영어 프롬프트의 정확도 차이는?

In [ ]:
# imagenette label은 코드처럼 생긴 것도 있어서 사람이 읽을 수 있게 매핑이 필요
READABLE = {
    'tench': 'tench fish',
    'English springer': 'English springer dog',
    'cassette player': 'cassette player',
    'chain saw': 'chain saw',
    'church': 'church building',
    'French horn': 'French horn',
    'garbage truck': 'garbage truck',
    'gas pump': 'gas pump',
    'golf ball': 'golf ball',
    'parachute': 'parachute',
}
READABLE_KO = {
    'tench': '잉어',
    'English springer': '스프링거 강아지',
    'cassette player': '카세트 플레이어',
    'chain saw': '전기톱',
    'church': '교회',
    'French horn': '프렌치 호른',
    'garbage truck': '쓰레기 수거차',
    'gas pump': '주유기',
    'golf ball': '골프공',
    'parachute': '낙하산',
}

# TODO: 영문 프롬프트 "a photo of a {READABLE[name]}" 10개
prompts_en = None
# TODO: 한국어 프롬프트 "{READABLE_KO[name]}의 사진" 10개
prompts_ko = None

# TODO: 두 버전 텍스트 임베딩 (encode_texts 사용)
text_feats_en = None
text_feats_ko = None

# TODO: image_feats 와의 유사도 argmax → 예측 → labels 와 비교해 accuracy 측정
#       (영어 vs 한국어 프롬프트 정확도 비교)

In [ ]:
# ✅ 참고 답안 (핵심)
prompts_en = [f'a photo of a {READABLE[name]}' for name in label_names]
prompts_ko = [f'{READABLE_KO[name]}의 사진' for name in label_names]
text_feats_en = encode_texts(prompts_en)
text_feats_ko = encode_texts(prompts_ko)

def zero_shot_accuracy(image_feats, text_feats, labels):
    preds = (image_feats @ text_feats.T).argmax(dim=1).numpy()
    return (preds == labels).mean(), preds

acc_en, _ = zero_shot_accuracy(image_feats, text_feats_en, labels)
acc_ko, _ = zero_shot_accuracy(image_feats, text_feats_ko, labels)
print(f'EN acc: {acc_en:.3f}  KO acc: {acc_ko:.3f}')   # 보통 EN ≳ KO

**관찰 포인트**
- 영어 프롬프트가 일반적으로 더 높은 정확도를 보임 (학습 데이터의 영어 비중이 높기 때문).
- 한국어 프롬프트도 상당히 높은 정확도 → SigLIP 2의 multilingual 능력 확인.
- 학습 데이터 0개로도 높은 zero-shot 정확도가 가능한 것이 CLIP/SigLIP 계열 모델이 가지는 강점.

## 선택 실습: Image → Image 유사도

텍스트가 아니라 *이미지를 쿼리*로 사용. 사진 갤러리에서 "이거랑 비슷한 사진" 찾기는 사실상 같은 코드. 두 모달리티 데이터 모두 같은 임베딩 공간에 있음.

### TODO #4 — 비슷한 이미지 찾기

갤러리에서 임의로 한 장 선택, 가장 유사한 다른 5장을 시각화해보세요.

같은 클래스가 retrieve되는지 확인해보시면 됩니다.

**힌트**: cosine similarity는 자기 자신이 1.0으로 가장 높음. 자기 인덱스 제외 처리 필요.

In [ ]:
import random; random.seed(7)
qi = random.randint(0, len(images) - 1)

# TODO: image_feats[qi] 와 모든 이미지 임베딩의 cosine similarity 계산
sims = None
# TODO: 자기 자신(qi) 제외 후 top-5 인덱스 구하기
top5 = None

# 시각화: 쿼리 이미지 + top-5 (TODO 를 채우면 동작합니다)
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(images[qi]); axes[0].axis('off')
axes[0].set_title(f'Query\n{label_names[labels[qi]]}', fontsize=8, color='red')
for ax, idx in zip(axes[1:], top5):
    idx = int(idx)
    ax.imshow(images[idx]); ax.axis('off')
    ax.set_title(f'{label_names[labels[idx]]}\n{sims[idx]:.3f}', fontsize=8)
fig.suptitle('Image → Image Retrieval (query + top-5)', y=1.02)
plt.tight_layout(); plt.show()

query_label = labels[qi]
same_class_count = sum(1 for idx in top5 if labels[int(idx)] == query_label)
print(f'쿼리 클래스: {label_names[query_label]}')
print(f'Top-5 중 같은 클래스: {same_class_count}/5')

In [ ]:
# ✅ 참고 답안 (핵심)
sims = image_feats[qi] @ image_feats.T   # (100,)
sims[qi] = -1.0                          # 자기 자신(=1.0) 제외
top5 = sims.argsort(descending=True)[:5]

---

## 정리

- **하나의 임베딩 모델**이 retrieval, 분류, 유사도 모두 담당
- 텍스트와 이미지가 **같은 공간**에 있어서 양방향 검색 가능
- SigLIP 2의 multilingual 학습 덕에 **한국어 쿼리도 동작**. 영어 프롬프트와의 격차가 어느 정도인지 측정
- 분류기 학습 데이터가 0개여도 **zero-shot accuracy**가 의외로 높음

## 다음 단계

이번 핵심 과정은 SigLIP 2 기반 검색·분류 실험으로 마무리합니다.